# 03. 한 약물에서 평균·유전자 Ridge·경로 Ridge 비교

약물 하나(Sorafenib)와 반복 하나(repeat 0)만 사용하는 작은 예제입니다. 몇 초 안에 끝납니다. 전체 실행은 노트북이 아니라 `sbatch experiments/pathway_sensitivity_a1/a2_internal.sbatch`로 합니다.

절차: 계열 단위로 5조각으로 나누고, 4조각으로 학습해 남은 1조각을 예측합니다. Ridge의 alpha는 학습 4조각 안에서 다시 3조각으로 나눠 고릅니다. 시험 조각의 정답은 채점에만 씁니다.

In [ ]:
import os, sys
root = os.getcwd()
while not os.path.isdir(os.path.join(root, "configs", "pathway_sensitivity_a1")):
    parent = os.path.dirname(root)
    assert parent != root, "저장소 안에서 실행해 주세요."
    root = parent
sys.path.insert(0, os.path.join(root, "src", "pathway_sensitivity_a1"))
os.environ.setdefault("OMP_NUM_THREADS", "1")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import core
cfg = core.load_config()
print("저장소:", root)
print("입력 캐시:", core.cache_dir(cfg))
import nested, metrics

In [ ]:
drug = "Sorafenib"
limore = core.load_study(cfg, "LIMORE")
sp = core.load_outer_splits(); sp = sp[sp.repeat == 0]
resp = core.load_limore_response(cfg)
models = list(sp.model_id)
rows = [limore.models.index(m) for m in models]          # 이름으로 정렬을 맞춥니다
Y = core.response_matrix(resp, models, [drug], "y")
fam, fold = sp.family_id.values, sp.outer_fold.values
X = {"A0": np.zeros((len(models), 1)), "A1": limore.expr[rows], "A3": limore.hallmark[rows]}
print("모델", len(models), " 계열", len(set(fam)), " fold별 모델 수", np.bincount(fold))

In [ ]:
pred, s_train = {}, np.zeros(len(models))
for kind in X:
    pred[kind] = np.zeros(len(models))
    for k, tr, te, p, info in nested.outer_folds(X[kind], Y, fam, fold, kind, cfg["model"]["alpha_grid"],
                                                 cfg["split"]["inner_folds"], lambda k: 7100 + k, K=50):
        assert not set(fam[tr]) & set(fam[te])            # 같은 계열이 학습과 시험에 함께 있지 않음
        pred[kind][te] = p[:, 0]
        s_train[te] = info["sd"][0]                      # 학습 조각의 표준편차
        if kind != "A0": print(kind, "fold", k, "선택된 alpha", info["alpha"][0])

## 채점

오차를 학습 표준편차로 나눈 뒤, 같은 계열 안에서 먼저 평균하고 계열 사이를 평균해 제곱근을 취합니다. 1.0 근처면 평균만 예측하는 것과 비슷하다는 뜻입니다.

In [ ]:
y = Y[:, 0]
for kind in X:
    z2 = pd.Series(((y - pred[kind]) / s_train) ** 2).groupby(fam).mean()
    yf, pf = pd.Series(y).groupby(fam).mean(), pd.Series(pred[kind]).groupby(fam).mean()
    print(kind, " 정규화 RMSE", round(float(np.sqrt(z2.mean())), 3), " Spearman", round(metrics.safe_spearman(yf, pf), 3))
print("주의: A0의 Spearman은 의미가 없습니다. 시험 조각을 뺀 평균이라 값이 거의 일정합니다.")

In [ ]:
fig, ax = plt.subplots(figsize=(4.4, 4))
for kind, c in (("A1", "#2a78d6"), ("A3", "#eda100")):
    ax.plot(y, pred[kind], "o", color=c, ms=5, label=kind)
ax.plot([y.min(), y.max()], [y.min(), y.max()], color="#52514e", lw=1, ls="--")
ax.set_xlabel("observed AA"); ax.set_ylabel("out-of-fold prediction"); ax.legend(frameon=False)